# Credit Logistic Regression Model

In [1]:
# packages
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.pyplot import subplots
import statsmodels.api as sm
from sklearn.model_selection import train_test_split 
from ISLP import load_data
from ISLP.models import (ModelSpec as MS, summarize)

# set seed
seed = 4721

In [2]:
# Load the data. Bureau and previous applications have many rows per loan, so roll them up to one row per SK_ID_CURR before joining.
application = pd.read_csv(
    "data/home-credit-default-risk/application_train.csv",
    usecols=[
        "SK_ID_CURR", "TARGET",
        "AMT_INCOME_TOTAL", "AMT_CREDIT", "AMT_ANNUITY", "NAME_CONTRACT_TYPE",
        "DAYS_BIRTH", "CODE_GENDER", "NAME_FAMILY_STATUS", "CNT_CHILDREN",
        "NAME_INCOME_TYPE", "DAYS_EMPLOYED",
        "NAME_EDUCATION_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
        "EXT_SOURCE_2", "EXT_SOURCE_3",
    ],
)
bureau = pd.read_csv(
    "data/home-credit-default-risk/bureau.csv",
    usecols=[
        "SK_ID_CURR", "CREDIT_ACTIVE", "CREDIT_DAY_OVERDUE",
        "AMT_CREDIT_SUM_OVERDUE", "AMT_CREDIT_SUM_DEBT",
    ],
)
previous = pd.read_csv(
    "data/home-credit-default-risk/previous_application.csv",
    usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS", "AMT_APPLICATION", "AMT_CREDIT"],
)

# Age and employment are stored as days before the application.
# 365243 marks people who are not employed, mostly pensioners.
application["DAYS_BIRTH"] = -application["DAYS_BIRTH"]
not_employed = application["DAYS_EMPLOYED"] == 365243
application["NOT_EMPLOYED"] = not_employed.astype(int)
application["DAYS_EMPLOYED"] = np.where(not_employed, 0, -application["DAYS_EMPLOYED"])
application["ANNUITY_TO_INCOME"] = application["AMT_ANNUITY"] / application["AMT_INCOME_TOTAL"]

# A handful of levels are too small for a stable logistic coefficient.
application = application[application["CODE_GENDER"] != "XNA"]
application = application[application["NAME_FAMILY_STATUS"] != "Unknown"]
rare_income = ["Unemployed", "Student", "Businessman", "Maternity leave"]
application["NAME_INCOME_TYPE"] = np.where(
    application["NAME_INCOME_TYPE"].isin(rare_income),
    "Other",
    application["NAME_INCOME_TYPE"],
)

bureau["IS_ACTIVE"] = (bureau["CREDIT_ACTIVE"] == "Active").astype(int)
bureau_by_loan = bureau.groupby("SK_ID_CURR").agg(
    BUREAU_N_ACTIVE=("IS_ACTIVE", "sum"),
    BUREAU_DAYS_OVERDUE=("CREDIT_DAY_OVERDUE", "max"),
    BUREAU_AMT_OVERDUE=("AMT_CREDIT_SUM_OVERDUE", "sum"),
    BUREAU_AMT_DEBT=("AMT_CREDIT_SUM_DEBT", "sum"),
).reset_index()

refused = previous["NAME_CONTRACT_STATUS"] == "Refused"
prev_refused = (
    previous.loc[refused]
    .groupby("SK_ID_CURR")
    .size()
    .rename("PREV_N_REFUSED")
    .reset_index()
)
prev_amounts = previous.groupby("SK_ID_CURR").agg(
    PREV_AMT_APPLICATION=("AMT_APPLICATION", "sum"),
    PREV_AMT_CREDIT=("AMT_CREDIT", "sum"),
).reset_index()
prev_amounts["PREV_AMT_GAP"] = (
    prev_amounts["PREV_AMT_APPLICATION"] - prev_amounts["PREV_AMT_CREDIT"]
)
prev_amounts = prev_amounts.drop(columns=["PREV_AMT_APPLICATION", "PREV_AMT_CREDIT"])

home_credit_data = (
    application
    .merge(bureau_by_loan, on="SK_ID_CURR", how="left")
    .merge(prev_refused, on="SK_ID_CURR", how="left")
    .merge(prev_amounts, on="SK_ID_CURR", how="left")
)
history_cols = [
    "BUREAU_N_ACTIVE", "BUREAU_DAYS_OVERDUE", "BUREAU_AMT_OVERDUE", "BUREAU_AMT_DEBT",
    "PREV_N_REFUSED", "PREV_AMT_GAP",
]
home_credit_data[history_cols] = home_credit_data[history_cols].fillna(0)
home_credit_data = home_credit_data.drop(columns=["AMT_ANNUITY"])

In [3]:
# List the columns and their types
home_credit_data.dtypes

SK_ID_CURR               int64
TARGET                   int64
NAME_CONTRACT_TYPE         str
CODE_GENDER                str
FLAG_OWN_CAR               str
FLAG_OWN_REALTY            str
CNT_CHILDREN             int64
AMT_INCOME_TOTAL       float64
AMT_CREDIT             float64
NAME_INCOME_TYPE           str
NAME_EDUCATION_TYPE        str
NAME_FAMILY_STATUS         str
DAYS_BIRTH               int64
DAYS_EMPLOYED            int64
EXT_SOURCE_2           float64
EXT_SOURCE_3           float64
NOT_EMPLOYED             int64
ANNUITY_TO_INCOME      float64
BUREAU_N_ACTIVE        float64
BUREAU_DAYS_OVERDUE    float64
BUREAU_AMT_OVERDUE     float64
BUREAU_AMT_DEBT        float64
PREV_N_REFUSED         float64
PREV_AMT_GAP           float64
dtype: object

In [5]:
indep_vars = [
    "AMT_INCOME_TOTAL", "AMT_CREDIT", "ANNUITY_TO_INCOME", "NAME_CONTRACT_TYPE",
    "DAYS_BIRTH", "CODE_GENDER", "NAME_FAMILY_STATUS", "CNT_CHILDREN",
    "NAME_INCOME_TYPE", "DAYS_EMPLOYED", "NOT_EMPLOYED",
    "NAME_EDUCATION_TYPE", "FLAG_OWN_CAR", "FLAG_OWN_REALTY",
    "EXT_SOURCE_2", "EXT_SOURCE_3",
    "BUREAU_N_ACTIVE", "BUREAU_DAYS_OVERDUE", "BUREAU_AMT_OVERDUE", "BUREAU_AMT_DEBT",
    "PREV_N_REFUSED", "PREV_AMT_GAP",
]

X = home_credit_data[indep_vars]
y = home_credit_data["TARGET"]

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    random_state=seed,
    test_size=0.25,
    shuffle=True,
)

# Category columns are strings, so corr() only uses the numeric features.
X_train.corr(numeric_only=True)

,AMT_INCOME_TOTAL,AMT_CREDIT,ANNUITY_TO_INCOME,DAYS_BIRTH,CNT_CHILDREN,DAYS_EMPLOYED,NOT_EMPLOYED,EXT_SOURCE_2,EXT_SOURCE_3,BUREAU_N_ACTIVE,BUREAU_DAYS_OVERDUE,BUREAU_AMT_OVERDUE,BUREAU_AMT_DEBT,PREV_N_REFUSED,PREV_AMT_GAP
AMT_INCOME_TOTAL,1.000000,0.334713,-0.319432,-0.054105,0.024316,0.081254,-0.134280,0.137652,-0.065549,0.128041,-0.006982,-0.000156,0.243664,0.042281,-0.076429
AMT_CREDIT,0.334713,1.000000,0.374450,0.055146,0.002400,0.102363,-0.065818,0.130594,0.044836,0.050147,-0.000216,-0.002868,0.103539,-0.040530,-0.034298
ANNUITY_TO_INCOME,-0.319432,0.374450,1.000000,0.081012,-0.001941,-0.014272,0.083374,-0.037140,0.104110,-0.139818,-0.000474,0.000463,-0.122032,-0.059367,0.061938
DAYS_BIRTH,-0.054105,0.055146,0.081012,1.000000,-0.329890,0.000274,0.619796,0.091113,0.204990,0.009155,0.012838,-0.003944,-0.069682,0.007921,-0.061490
CNT_CHILDREN,0.024316,0.002400,-0.001941,-0.329890,1.000000,0.042604,-0.240941,-0.016830,-0.040198,0.015188,-0.002441,-0.000539,0.049375,-0.016581,0.028968
DAYS_EMPLOYED,0.081254,0.102363,-0.014272,0.000274,0.042604,1.000000,-0.396579,0.084599,0.062054,0.097432,0.003563,-0.000680,0.072619,-0.008317,-0.044059
NOT_EMPLOYED,-0.134280,-0.065818,0.083374,0.619796,-0.240941,-0.396579,1.000000,-0.020330,0.115909,-0.040191,0.007703,-0.004968,-0.090428,-0.001725,0.016420
EXT_SOURCE_2,0.137652,0.130594,-0.037140,0.091113,-0.016830,0.084599,-0.020330,1.000000,0.110925,0.012196,-0.003431,-0.002013,0.058307,-0.035256,-0.011772
EXT_SOURCE_3,-0.065549,0.044836,0.104110,0.204990,-0.040198,0.062054,0.115909,0.110925,1.000000,-0.393206,-0.030498,-0.026529,-0.124913,-0.187153,0.082674
BUREAU_N_ACTIVE,0.128041,0.050147,-0.139818,0.009155,0.015188,0.097432,-0.040191,0.012196,-0.393206,1.000000,0.034272,0.012857,0.323719,0.151202,-0.117579
